# Football-Player-Tracker — GitHub-first Colab MOT Export (v3)

**Fix:** the previous notebook repeatedly installed/upgraded packages and imported SciPy in the same runtime, causing `numpy._core.umath: _slice` errors. This notebook installs pinned binary wheels, **requires a runtime restart**, and verifies compiled imports in a fresh process **before** importing BoxMOT. It does **not** patch repository source in Colab.

**One-time prerequisite:** apply the accompanying `migrate_boxmot25.py` script in your **local GitHub checkout**, review the diff, commit, and push to `main`. It removes the old BoxMOT path forcing, updates `src/tracker.py`, and makes `configs/project_config.py` use its repo directory with a Drive output override. The notebook deliberately refuses to run tracking with unmigrated source. GitHub `main` must also have `gmc_method: sof`.

**Run order:** A → **restart runtime manually** → B → C → D → E → F → G. Do not re-run A after restart unless packages need repairing. Colab GPU runtime recommended.

## Cell A — Install fixed dependencies (RUN ONCE, THEN RESTART RUNTIME)

In [ ]:
import sys, subprocess

# Keep NumPy/SciPy in the same dependency transaction as BoxMOT and Ultralytics.
# Avoid unconstrained `pip install scipy boxmot ...` and repeated in-notebook upgrades.
packages = [
    'numpy==2.3.3',
    'scipy==1.16.2',
    'boxmot==25.0.0',
    'ultralytics==8.4.183',
    'opencv-python-headless',
    'pandas', 'pyyaml', 'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '--upgrade-strategy', 'only-if-needed', *packages], check=True)
# Replace any mixed NumPy/SciPy Python/binary files (without disturbing Torch).
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '--force-reinstall', '--no-deps', '--only-binary=:all:',
                'numpy==2.3.3', 'scipy==1.16.2'], check=True)

# Verify the *on-disk* packages in a fresh process, not cached notebook imports.
probe = "import numpy as n, scipy as s, scipy.linalg; print('NumPy',n.__version__,'SciPy',s.__version__); assert n.__version__=='2.3.3' and s.__version__=='1.16.2'; print('det',scipy.linalg.det(n.eye(2)))"
subprocess.run([sys.executable, '-c', probe], check=True)
print('INSTALL COMPLETE. REQUIRED: Runtime > Restart session. Then continue at Cell B.')


## Cell B — After restarting: verify compiled imports (no installation)

In [ ]:
import sys, subprocess, importlib.metadata
probe = r"""
import numpy as np
import scipy
import scipy.linalg
import cv2
from boxmot.trackers.box.botsort.tracker import BotSort
assert np.__version__ == '2.3.3', np.__version__
assert scipy.__version__ == '1.16.2', scipy.__version__
print('NumPy',np.__version__,'SciPy',scipy.__version__)
print('SciPy det',scipy.linalg.det(np.eye(2)))
print('OpenCV',cv2.__version__, 'BotSort import successful')
"""
subprocess.run([sys.executable, '-c', probe], check=True)
print('All compiled imports passed. No need to reinstall or restart again.')


## Cell C — Mount Drive and refresh GitHub **without editing source**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os
DRIVE_ROOT = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'Utilities' / 'Football-Player-Tracker'
PROJECT_CODE_DIR = Path('/content/Football-Player-Tracker')
REPO_URL = 'https://github.com/Alejandro-Fuste/Football-Player-Tracker.git'
CLIPS_DIR = DRIVE_ROOT / 'Datasets' / 'AmericanFootballDataset' / 'videos'
EXPORT_DIR = DRIVE_PROJECT_DIR / 'exports'
CSV_PATH = DRIVE_PROJECT_DIR / 'ToBeTracked.csv'
if not DRIVE_PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Create Drive output directory: {DRIVE_PROJECT_DIR}')
if PROJECT_CODE_DIR.exists():
    subprocess.run(['git','-C',str(PROJECT_CODE_DIR),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT_CODE_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','--branch','main',REPO_URL,str(PROJECT_CODE_DIR)],check=True)
commit = subprocess.check_output(['git','-C',str(PROJECT_CODE_DIR),'rev-parse','HEAD'],text=True).strip()
os.environ['FOOTBALL_TRACKER_EXPORT_DIR']=str(EXPORT_DIR)
os.chdir(PROJECT_CODE_DIR)
if str(PROJECT_CODE_DIR) not in sys.path: sys.path.insert(0,str(PROJECT_CODE_DIR))
print('GitHub commit:',commit)
print('Source:',PROJECT_CODE_DIR,'\nVideos:',CLIPS_DIR,'\nExports:',EXPORT_DIR)


## Cell D — Repository migration preflight (no monkeypatches)

In [ ]:
import ast, yaml, os
tracker_path = PROJECT_CODE_DIR/'src/tracker.py'
script_path = PROJECT_CODE_DIR/'scripts/detect_track_export.py'
project_config_path = PROJECT_CODE_DIR/'configs/project_config.py'
tracker_yaml = PROJECT_CODE_DIR/'configs/football_botsort.yaml'
for p in [tracker_path, script_path, project_config_path, tracker_yaml]:
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix == '.py': ast.parse(p.read_text())
failures=[]
if '/tmp/boxmot_repo' in script_path.read_text():
    failures.append('scripts/detect_track_export.py still contains legacy BoxMOT import forcing')
if 'from boxmot.trackers.box.botsort.tracker import BotSort' not in tracker_path.read_text():
    failures.append('src/tracker.py has not been migrated to the BoxMOT 25 direct BotSort adapter')
if 'FOOTBALL_TRACKER_EXPORT_DIR' not in project_config_path.read_text():
    failures.append('configs/project_config.py needs GitHub-source/Drive-export migration')
cfg=yaml.safe_load(tracker_yaml.read_text())
if cfg.get('gmc_method') != 'sof':
    failures.append(f"configs/football_botsort.yaml has gmc_method={cfg.get('gmc_method')!r}, should be 'sof'")
if failures:
    raise RuntimeError('GitHub main requires migration; apply migrate_boxmot25.py locally, commit and push, then rerun Cell C:\n - '+'\n - '.join(failures))
from configs import project_config
assert project_config.PROJECT_ROOT.resolve() == PROJECT_CODE_DIR.resolve()
assert project_config.EXPORT_DIR.resolve() == EXPORT_DIR.resolve()
assert CLIPS_DIR.is_dir(), f'Videos directory not found: {CLIPS_DIR}'
assert (PROJECT_CODE_DIR/'models/osnet_x0_25_msmt17.pt').is_file()
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
print('GitHub tracker and configuration passed. Camera motion:',cfg['gmc_method'])
registry=yaml.safe_load((PROJECT_CODE_DIR/'configs/model_registry.yaml').read_text())
for name, spec in registry.get('models',{}).items():
    p=Path(spec.get('weights_path',''))
    print('Detector model:',name,'weights exist:',p.is_file(),p)


## Cell E — Choose manifest and initialize tracker with one video frame

In [ ]:
import csv, cv2, numpy as np
from src.tracker import create_botsort_tracker
if not CSV_PATH.is_file():
    CSV_PATH = PROJECT_CODE_DIR/'ToBeTracked.csv'
if not CSV_PATH.is_file(): raise FileNotFoundError('ToBeTracked.csv missing in Drive or GitHub repo')
with CSV_PATH.open(newline='',encoding='utf-8-sig') as f:
    reader=csv.DictReader(f)
    field=next((k for k in (reader.fieldnames or []) if k.strip().lower() in ('filename','shortname')),None)
    if field is None: raise ValueError('CSV needs FileName or ShortName header')
    names=[row[field].strip() for row in reader if row.get(field,'').strip()]
if not names: raise ValueError('Manifest is empty')
def find_clip(name):
    p=CLIPS_DIR/name
    if p.is_file(): return p
    for ext in ('.mp4','.MP4','.mov','.MOV','.avi','.AVI','.mkv','.MKV'):
        q=CLIPS_DIR/(Path(name).stem+ext)
        if q.is_file(): return q
    raise FileNotFoundError(f'Missing clip: {name}')
missing=[]
for name in names:
    try: find_clip(name)
    except FileNotFoundError: missing.append(name)
if missing: raise FileNotFoundError(f'Missing {len(missing)} video(s): {missing[:20]}')
print('Manifest:',CSV_PATH,'\nClip count:',len(names))
first_path=find_clip(names[0]); cap=cv2.VideoCapture(str(first_path))
try: ok,frame=cap.read()
finally: cap.release()
if not ok: raise RuntimeError(f'Cannot read video frame: {first_path}')
tracker=create_botsort_tracker(tracker_yaml)
out=tracker.update(np.empty((0,6),dtype=np.float32),frame)
print('BoT-SORT empty-frame update:',np.asarray(out).shape)


## Cell F — 5-frame end-to-end CVAT MOT ZIP smoke test

In [ ]:
import subprocess,sys,zipfile
SMOKE_OUT=EXPORT_DIR/'cvat_mot_tracks'/'smoke_test'/f'{first_path.stem}_cvat_mot.zip'
SMOKE_OUT.parent.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'scripts/detect_track_export.py',
    '--video',str(first_path),'--model-config',str(PROJECT_CODE_DIR/'configs/model_registry.yaml'),
    '--model-key','rtdetrx_player_best','--tracker-config',str(tracker_yaml),
    '--out',str(SMOKE_OUT),'--conf','0.64','--max-frames','5']
subprocess.run(cmd,check=True,cwd=PROJECT_CODE_DIR,env=os.environ.copy())
assert SMOKE_OUT.is_file() and SMOKE_OUT.stat().st_size>0
with zipfile.ZipFile(SMOKE_OUT) as z:
    print('MOT ZIP members:',z.namelist())
    assert any(p.endswith('gt.txt') for p in z.namelist())
print('5-frame smoke test passed:',SMOKE_OUT)


## Cell G — Reclipped-video batch, then mandatory failure audit

In [ ]:
import json
BATCH_NAME='Reclipped_Tracking'
BATCH_OUT=EXPORT_DIR/'cvat_mot_tracks'/'batch'/BATCH_NAME
cmd=[sys.executable,'scripts/batch_export.py',
    '--model-config',str(PROJECT_CODE_DIR/'configs/model_registry.yaml'),
    '--model-key','rtdetrx_player_best','--tracker-config',str(tracker_yaml),
    '--clips-dir',str(CLIPS_DIR),'--out-dir',str(BATCH_OUT),
    '--csv-list',str(CSV_PATH),'--conf','0.64','--save-preview','--write-summary']
result=subprocess.run(cmd,cwd=PROJECT_CODE_DIR,env=os.environ.copy())
summary_path=BATCH_OUT/'batch_tracking_summary.json'
if not summary_path.is_file(): raise RuntimeError(f'Batch summary missing; exit={result.returncode}')
items=json.loads(summary_path.read_text())
for row in items: print(row.get('video_name',row.get('video')),row.get('status'),row.get('error_message',''))
failed=[row for row in items if row.get('status')=='failed']
if failed:
    for row in failed:
        log=BATCH_OUT/'logs'/(Path(row.get('video_name','unknown')).stem+'_error.log')
        if log.exists(): print(log.read_text()[-6000:])
    raise RuntimeError(f'Batch has {len(failed)} failed clip(s).')
if result.returncode: raise RuntimeError(f'Batch process exit code {result.returncode}')
print('Batch completed. Entries:',len(items),'Output:',BATCH_OUT)
